# IPySlides Demo
This demo loads the most recent development version of `ipyslides` package in binder and installed version elsewhere. Run cell by cell to keep updating slides.

**Tip:**
> Use `Create New View for Output` on next cell (after run) to view slides on slide while keep adding slides by cells run.

In [ ]:
import ipyslides as isd 

slides = isd.Slides(
    layout = dict(scroll = True, aspect=16/10),
)
# Apply these settings or change later in many flexible ways
slides.settings.layout.scroll = True # As assignment
slides.settings.logo(src=slides.get_logo(), width=25) # As call
slides.settings.footer.text = slides.get_logo("1em") + 'Author: Abdul Saboor عبدالصبور'
slides.set_citations(r'''
    @pf: This is refernce to FigureWidget using [alert: \@pf /] syntax
    @This: I was cited for no reason
''') # set citations early to be used across slides
slides

Below is title slide with number `0` and markdown flag `-m`.

In [ ]:
%%slide 0 -m
```md-src.collapsed
# Creating Slides
::: align-center width=50%
    [alert: Abdul Saboor /][sup:1/], Unknown Author[sup:2/]
    [center: [today:/] /]
    ::: align-left text-box
        [sup:1/]My University is somewhere in the middle of nowhere
        [sup:2/]Their University is somewhere in the middle of nowhere

::: display align-center               
    vspace`2`Right click (or click on footer) to open context menu and click on fa`info` icon for instructions.
```
[md-src/]

`-1` in below slide will automatically change to slide number in the order cells run by user. Note that two slides being built from a single cell.

In [ ]:
with slides.slide(-1): slides.src("section`Introduction` toc`### Contents`")

with slides.slide(-1): 
    slides.src(f"""
    # Introduction
                  
    To see how commands work, use code`ipyslides.docs()` to see the documentation.
    Here we will focus on using some of that functionality to create slides.
    
    ::: note-info
        This slide was built purely from markdown, so you can create
        a variable `test` to overwite this → %{{test}}
                  
    Version: `{slides.version}`
    """)

In below slide `btn` is a widget inserted in markdown.

In [ ]:
%%slide -1
slides.src("""
::: md-after
    [section: Adding informative TOC /]
    ```columns .block-blue
    [toc: ### Contents :: True /]
    +++
    vspace`1` This is summary for current section created using block syntax of toc. See `Slides.xmd.syntax` for details.
    - Item 1
    - Item 2
    $$ E = mc^2 $$ 
    %{btn}  
    ::: note-tip
        Above `btn` variable can be updated later via `Slides[number,].vars.update` method.                 
    ```                                
""",btn=slides.draw_button)

Here is a lazy slide (by using `@src` decorator) that will run on a button click on bottom right of slides.

In [ ]:
%%slide -1
slides.src("I will be overwritten by function call below lazily!")

@slides.src
def _(sl):
    slides.write('## Plotting with Matplotlib section`Plotting and DataFrame`')
    with slides.code.context(returns = True) as s:
        slides.css(bg1 = 'linear-gradient(to right, #FFDAB9 0%, #F0E68C 100%)')
        import numpy as np, matplotlib.pyplot as plt
        plt.rcParams['svg.fonttype'] = 'none' # Global setting, enforce same fonts as presentation
        x = np.linspace(0,2*np.pi)
        with plt.style.context('ggplot'):
            fig, ax = plt.subplots(figsize=(3.4,2.6))
            _ = ax.plot(x,np.cos(x))
        slides.write(ax, s.focus([0,3,4]))
        slides.write('Double click on the plot to focus on it!')

In [ ]:
%%slide -1
# dependent code can be run here before declaring src function
with slides.code.context(returns = True) as source:
    try:
        import pandas as pd 
        df = pd.read_csv('https://raw.githubusercontent.com/mwaskom/seaborn-data/master/iris.csv')
        df = df.describe() #Small for display
    except:
        df = '### Install `pandas` to view output'

@slides.src
def _(s):
    slides.write(['## Writing Pandas DataFrame', df, source]) # lazy

In [ ]:
with slides.code.context(returns = True) as source:
    try:
        import ipywidgets as ipw
        import plotly.graph_objects as go
        fig = slides.patched_plotly(go.FigureWidget()) # prefere Widget for interactivity and correct display
        fig.add_trace(go.Bar(y=[1,5,8,9], customdata=["A","B"]))
        
        # We have clicked and selected traits on patched plotly
        html = ipw.HTML()
        def observe_click(change):
            html.value = "<br/>".join(f"  {k} = {v}" for k, v in change['new'].items())
        fig.observe(observe_click, names='clicked')
        box = ipw.HBox([fig, html])
    except:
        box = '### Install `plotly` to view output'

with slides.slide(-1):
    @slides.src
    def _(s):
        slides.write(['## Writing Plotly Figure',box, source]) # lazy for no reason 🤣

In [ ]:
# Resuable functions
def race_plot():
    import numpy as np
    import matplotlib.pyplot as plt
    x = np.linspace(0,0.9,10)
    y = np.random.random((10,))
    _sort = np.argsort(y)
    plot_theme = 'dark_background' if 'Dark' in slides.settings.layout.theme else 'default'
    with plt.style.context(plot_theme):        
        fig,ax = plt.subplots(figsize=(3.4,2.6))
        ax.barh(x,y[_sort],height=0.07,color = plt.colormaps['plasma'](x[_sort]))
    for s in ['right','top','bottom']:
        ax.spines[s].set_visible(False)
    ax.set(title='Race Plot', ylim = [-0.05,0.95], xticks=[],yticks=[c for c in x],yticklabels=[rf'$X_{int(c*10)}$' for c in x[_sort]])
    return slides.html("div", slides.plt2html(fig, transparent=False, caption='A Silly Plot').value, css_class="anim-wipe-down") # class auto triggers on chnaging content

In [ ]:
with slides.slide(-1) as rslide:
    slides.write('''
        ## Refreshable Content
        Use refresh button below to update plot!
        See alert`race_plot` function at end of slides.
        ''')
    
    def display_plot(btn): return race_plot().display()
    
    slides.write(
        slides.dl.interactive(display_plot, btn = slides.dl.button('Refresh Plot', icon='refresh')), 
        rslide.get_source()
    ) # Only first columns will update

In [ ]:
with slides.slide(-1) as s:
    slides.write('## Animations with Widgets')
    anim = slides.AnimationSlider(nframes=20, interval=1500, continuous_update=False)
    css = {'grid-template-columns': '1fr 2fr', '.out-main': {'height': '2em'}}
    
    @slides.dl.interact(post_init = lambda self: self.set_css(center=css), html = slides.as_html_widget(''), source=s.get_source().as_widget(), anim=anim)
    def _(html, source, anim):
        html.value = race_plot().value
        print(f'Animation Frame: {anim}') # goes to output area

### A full interactaive dashboard on slides

In [ ]:
%%slide -1
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import plotly.graph_objects as go

def on_click(cdata,html):
    html.value = pd.DataFrame(cdata or {}).to_html(index=False)

def on_select(sdata, html):
    plt.scatter(sdata.get('xs',[]),sdata.get('ys',[]))
    html.value = slides.plt2html().value


@slides.dl.interact(
    on_select,
    on_click,
    fig = slides.dl.patched_plotly(go.FigureWidget()), 
    html = slides.as_html_widget('**Select Box/Lesso on figure traces**'),
    A = (1,10), omega = (0,20), phi = (0,10),
    sdata = 'fig.selected', cdata = 'fig.clicked',
    post_init = lambda self: (
        self.set_css({'.left-sidebar':{'background':'whitesmoke'}}),
        self.set_layout(
            left_sidebar=['A','omega','phi','html'], 
            center=['fig']
        )
    )
)
def plot(fig:go.FigureWidget, A, omega,phi): # adding type hint allows auto-completion inside function
    fig.data = []
    x = np.linspace(0,10,100)
    fig.add_trace(go.Scatter(x=x, y=A*np.sin(omega*x + phi), mode='lines+markers'))

In [ ]:
%%slide -1
@slides.src
def _(s):
    import numpy as np
    import matplotlib.pyplot as plt
    def plot_sine():
        plt.plot(np.sin(np.linspace(0,10,100)))
    lw = slides.ListWidget(description='Execute a code block',
        options = [
            lambda: print(np.random.random((10,2))),
            lambda: plt.plot(np.random.random((10,2))),
            plot_sine,
        ], transform = lambda value: slides.code(value).inline.value # need simple code, otherwise defult transform is fine
    )
    def run(c):
        if callable(c): c() # avoid None value when not selected
        plt.show()
    
    css = {'.out-main': {'height':'300px'}, 'grid':'auto-flow / 1fr', '.lang-name': {'display': 'none'}} # just single column
    it = slides.dl.interactive(run, c = lw, post_init = lambda self: self.set_css(css)) 
    slides.write(['### Rich Content code`ListWidget`', it],s.get_source())

**Inplace minipages-like content (exportable unlike programmatic animations)**

In [ ]:
with slides.slide(-1): 
    slides.src('section`Simple Animations with Frames` toc`### Contents`')

In [ ]:
%%slide -1
@slides.src
def _(s):
    slides.write(
        "## Animating Matplotlib!", 
        slides.link('frames-end', 'Skip All Next Frames', icon='arrowr', uid='frames-start')
    )
    
    with slides.code.context(returns = True) as src:
        import numpy as np
        import matplotlib.pyplot as plt
        plots = slides.group(snapshots=True) # create empty snapshots-like group to append frames to later
        for idx in range(10,19):
            _, ax = plt.subplots(figsize=(3.4,2.6))
            x = np.linspace(0,idx,50)
            ax.plot(x,np.sin(x))
            ax.set_title(rf'$f(x)=\sin(x)$, 0 < x < {idx+1}')
            ax.set_xlim([0,18])
            ax.set_axis_off()
            plots.append(ax)
    # Fixed source at left + row-wise reveals of plots at right.
    slides.pause()
    slides.write(src, plots, widths=[60,40])
    slides.write('Unlike `interact/interactive`, this animation is based on slide frames, all of which are exported to HTML.',css_class='note-tip')

In [ ]:
%%slide -1
import matplotlib.pyplot as plt
from functools import wraps

with slides.code.context(True) as sc:
    @slides.xmd.register("plot")
    @wraps(plt.plot)
    def _(*args, **kwargs):
        plt.plot(*args, **kwargs)
        cap="Matplotlib [alert: in Markdown! /]"
        # We need crisp html of plots
        return slides.plt2html(caption = cap)

slides.src('''
```md-src
::: columns 3 2 2
    ## Inline Python Functions
    
    [plot! [1,2,7] /]   
    [plot! [1,5,3] /] 

[stack: %{sc} || [md-src/] /]
```
''', sc= sc)

In [ ]:
with slides.slide(-1): 
    slides.src('section`Controlling Content on Frames` toc`### Contents`')

# Frames structure
boxes = [slides.html('h1', f"{c}",style="background:var(--bg3-color);margin-block:0.05em !important;") for c in range(1,5)]

with slides.slide(-1) as s:
    slides.write('# Frames with Snapshots yoffset`0`')
    s.get_source().focus([2,3,4]).display()
    icons = {1:'edit', 2:'gear', 3:'download', 4:'trash'} # just for fun, can be any marker or None
    slides.pause(isolate=True) # isolate to split from previous content
    slides.write(slides.group(boxes, 
        points=True, snapshots=True, 
        marker=lambda idx:f"[fa:{icons[idx]} :: 'red'/]",
        css_class='anim-group anim-slide-left', # per item animation via parent group
    )) # snapshots reveal rows in isolation

In [ ]:
with slides.slide(-1) as s:
    slides.write('# Frames with \n#### code`pause.iter()` and 2x2 grid of boxes')
    s.get_source().focus(range(2,7)).display()
    objs = [boxes[:2],boxes[2:]]
    widths = [(1,3),(3,2)]
    for ws, cols in slides.pause.iter(zip(widths,objs), isolate=True): # isolate to split from previous content
        slides.write(*cols, widths=ws, css_class='anim-group anim-wipe-right')

In [ ]:
# Youtube
from IPython.display import YouTubeVideo
with slides.slide(-1) as ys: # We will use this in next %%magic
    slides.link('frames-start', 'Skip Previous Frames', 
        icon='arrowl', uid='frames-end').display()
    
    slides.xmd(f"### Watching Youtube Video?")
    slides.write('**Want to do some drawing instead?** Click on button on the right!', slides.draw_button, widths=[3,1])
    slides.write(
        YouTubeVideo('thgLGl14-tg',width='100%',height='266px'),
        YouTubeVideo('XZt-lH8Za3Q',width='100%',height='266px'),
    )
    @slides.on_load
    def push(slide):
        import time
        t = time.localtime()
        slides.notify(f'You are watching Youtube on slide {slide.index} at Time-{t.tm_hour:02}:{t.tm_min:02}')
    ys.get_source().display(True) 

In [ ]:
with slides.slide(-1) as s:
    import ipywidgets as ipw
    slides.xmd('## Blocks with CSS classes')
    slides.write(
        [
            '### Table',
            '''
            ::: table 1 2 1 block-green width=100%
                |h1 |h2 |h3 |
                |---|---|---|
                |d1 |d2 |d3 |
                |r1 |r2 |r3 |
            
            line`200`
            ### A rich content table
            ''',
            slides.table([[slides.icon('loading'), 2,3],[3,'code`import numpy as np`', 5]],headers=['h1','h2','h3'],widths=[3,5,1]),
        ], 
        [
            '### Widgets',
            slides.alt(lambda w: f'<input type="range" min="{w.min}" max="{w.max}" value="{w.value}">', ipw.IntSlider()), 
            slides.alt('<button>Click to do nothing</button>', ipw.Button(description='Click to do nothing')), 
            ipw.Checkbox(description='Select to do nothing',indent=False), 
        ], css_class="block-red"
    )
    s.get_source().display(True)

In [ ]:
%%slide -1
slides.src(r'''
## $ \LaTeX $ in Slides
```columns
Use `$ $` or `$$ $$` to display latex in Markdown, or embed images of equations
$ \LaTeX $ needs time to load, so keeping it in view until it loads would help.
{.info}
                            
                    
::: note-tip
    Varibale formatting alongwith $ \LaTeX $ alert`\%{var} → %{var}` is seamless.
```
::: md-src.collapsed
    ++[isolate]
    ```columns 50 50 anim-group anim-slide-up
    $$ \int_0^1\\frac{1}{1-x^2}dx $$
    {.align-left .text-big .info}
    +++
    ::: success
        $$ ax^2 + bx + c = 0 $$
        {.text-huge}
    ++
    [pin! 10,30, rotate=15 ::
        ::: anim-bounce info | Hello 👋, I am pinned here! 
    /]
    ```
[md-src/]
''', var = "'I was a variable'")

In [ ]:
with slides.slide(-1) as some_slide:
    slides.write('## Serialize Custom Objects to HTML\nThis is useful for displaying user defined/third party objects in slides section`Custom Objects Serilaization`')
    with slides.suppress_stdout(): # suppress stdout from register fuction below
        @slides.serializer.register(int)
        def colorize(obj):
            color = 'red' if obj % 2 == 0 else 'green'
            return f'<span style="color:{color};">{obj}</span>'
        slides.write(*range(10))

    some_slide.get_source().display()

#### Backup slides with `[section: Backup Slides :: True /]`

In [ ]:
%%slide -1
slides.xmd('[section: Code to Generate Slides :: True /]')
slides.get_source().display()
slides.navigate_to(0) # Go to title slide 